# Базовые операции с геометрией

Геообработка — это набор инструментов, используемых для анализа и преобразования геопространственных данных. Эти инструменты позволяют:

- выполнять пространственные операции;
- анализировать взаимное расположение объектов;
- создавать новые наборы данных на основе существующих.

В этом разделе мы рассмотрим несколько наиболее часто используемых базовых инструментов работы с геометрией:

- **Буферизация (Buffer)** — создание зон вокруг объектов на заданном расстоянии;
- **Обрезка (Clip)** — отсечение одного набора данных по границам другого;
- **Агрегация (Dissolve)** — объединение геометрий на основе общего атрибута.


## 0. Импорт библиотек и подготовка данных


### 0.1 Импорт библиотек


In [ ]:
import osmnx as ox

### 0.2 Подготовка данных


Загрузим границу Центрального района Санкт-Петербурга


In [ ]:
area_name = "Центральный район, Санкт-Петербург"
admin_border = ox.geocode_to_gdf(area_name)

admin_border.explore(tiles="cartodbpositron")

Также загрузим точки автобусных остановок и входов в метро


In [ ]:

tags = {
    "highway": "bus_stop",
    "railway": ["subway_entrance"]
}

stops = ox.features_from_place(
    area_name,
    tags
)

stops.explore(tiles="cartodbpositron")

Рассмотрим структуру полученных данных и выберем поля, которые будут использоваться в дальнейшем анализе.


In [ ]:
stops.head()

In [ ]:
stops["osm_id"] = stops.index.get_level_values("id")
stops = stops[["osm_id", "railway", "name", "highway", "geometry"]]
stops.head()

Теперь данные подготовлены, и можно переходить к пространственному анализу.


## 1. Буферизация (Buffering)

Буферизация — это создание области вокруг геометрических объектов на заданном расстоянии.

Такая операция позволяет формировать зоны вокруг объектов, которые можно использовать для дальнейшего пространственного анализа. Например, с её помощью можно определить территории в пределах заданного радиуса или выделить объекты, расположенные поблизости.

Буфер создаётся с помощью метода `.buffer()`, где расстояние задаётся в единицах системы координат.

Если данные находятся в **географической системе координат** (широта/долгота), расстояние буфера будет интерпретироваться в **градусах**, а не в метрах. Именно поэтому важно заранее проверять систему координат и при необходимости перепроецировать данные.

> Корректное использование буфера возможно только в **проецированной системе координат**, где расстояния выражены в метрах.


Создадим буферные зоны радиусом 500 метров вокруг остановок общественного транспорта.

Перед созданием буфера необходимо убедиться, что данные находятся в системе координат с метрическими единицами измерения.


### 1.1. Проверка системы координат и перепроецирование


Проверим систему координат исходных данных


In [ ]:
stops.crs

Данные находятся в географической системе координат. Нам это не подходит для дальнейшего анализа. Перепроецируем данные в подходящую UTM-зону


In [ ]:
data_crs = stops.estimate_utm_crs()

stops_utm = stops.to_crs(data_crs)

Проверим систему координат после перепроецирования


In [ ]:
stops_utm.crs

Теперь единицы измерения у системы координат в метрах, и мы можем переходить к созданию буферной зоны


### 1.2. Создание буфера


In [ ]:
# создаем копию GeoDataFrame с остановками
stops_buffer = stops_utm.copy()

# строим буфер радиусом 500 метров вокруг каждой остановки
stops_buffer["geometry"] = stops_buffer.geometry.buffer(500)

# проверяем результат
stops_buffer.explore(tiles="cartodbpositron")

После создания буферов отдельные зоны могут перекрываться. Чтобы получить единую зону доступности, необходимо объединить их.


## 2. Агрегация (Dissolve)

Агрегация — это объединение нескольких геометрических объектов в один на основе заданного признака или без него. В GeoPandas эта операция выполняется с помощью метода `.dissolve()`.

Данная операция позволяет:

- объединять объекты в более крупные территориальные единицы;
- агрегировать данные по группам (например, по районам);
- формировать единую геометрию из множества отдельных объектов.


Создадим общую зону 500-метровой доступности остановок общественного транспорта


Если метод `.dissolve()` используется без указания поля, все геометрии объединяются в один объект.


In [ ]:
stops_dissolved_all = stops_buffer.dissolve()

stops_dissolved_all.explore(tiles="cartodbpositron")

Если необходимо выполнить агрегацию по определённому признаку (например, по типу остановки), необходимо указать соответствующее поле.

В нашем случае создадим две разные зоны доступности: для остановок наземного общественного транспорта и для станций метро. Для этого необходимо разделить объекты по типу транспорта.

В исходных данных можно использовать поле `railway`: для станций метро в нём указано значение `subway_entrance`, а для остальных объектов значения отсутствуют (`NaN`). Однако такой подход не всегда удобен для группировки, поэтому создадим отдельное поле с типом транспорта.


In [ ]:
stops_buffer["transport_type"] = "bus"

stops_buffer.loc[
    stops_buffer["railway"] == "subway_entrance",
    "transport_type"
] = "subway"

Теперь можно выполнить агрегацию по новому полю:


In [ ]:
stops_dissolved_cat = stops_buffer.dissolve(by="transport_type")
stops_dissolved_cat.explore(tiles="cartodbpositron")

В результате мы получили две зоны доступности — отдельно для станций метро и остановок наземного общественного транспорта.


## 3. Обрезка (Clip)

Обрезка — это операция, позволяющая выделить части геометрий, которые находятся внутри заданной области. По сути, она «обрезает» объекты по границе другого слоя, оставляя только ту часть, которая попадает внутрь него.

Эта операция особенно полезна, когда необходимо ограничить результаты анализа определённой территорией.

В нашем случае после построения буферов и их агрегации необходимо оставить только ту часть зон доступности, которая находится внутри рассматриваемого района.

При использовании данного метода важно проверять, что используемые наборы данных находятся в одной системе координат.


Обрежем зоны доступности так, чтобы они не выходили за пределы границы Центрального района


Проверим систему координат у набора данных с границами района и перепроецируем её при необходимости


In [ ]:
admin_border.crs == stops_dissolved_cat.crs

Системы координат не совпадают. Перепроецируем границы района в ту же систему координат, в которой находятся остановки.


In [ ]:
admin_border_utm = admin_border.to_crs(stops_dissolved_cat.crs)

Обрежем по ним зоны доступности и посмотрим на результат


In [ ]:
stops_clipped = stops_dissolved_cat.clip(admin_border_utm)

stops_clipped.explore(tiles="cartodbpositron")

В результате мы получаем зоны доступности, ограниченные границами района исследования.


## Итог


В этом разделе мы познакомились с базовыми инструментами работы с геометрией и применили их для анализа доступности общественного транспорта.

Мы узнали:

- как создавать буферные зоны с помощью `.buffer()`;
- как обрезать данные по границам с помощью `clip()`;
- как объединять геометрии по атрибутам с помощью `.dissolve()`;
